# Lab 2: Querying RDF graphs using SPARQL and algorithms for data matching



In [1]:
## Installation of libraries
#Install the necessary libraries, run only once: 
import sys
!{sys.executable} -m pip install sparqlwrapper
!{sys.executable} -m pip install rdflib
!{sys.executable} -m pip install -U pip setuptools wheel
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download en_core_web_lg
!{sys.executable} -m pip install pandas numpy 

  Using cached SPARQLWrapper-2.0.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached rdflib-7.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached SPARQLWrapper-2.0.0-py3-none-any.whl (28 kB)
Using cached rdflib-7.6.0-py3-none-any.whl (615 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sparqlwrapper]0m [rdflib]
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.9.0
    Uninstalling setuptools-80.9.0:
      Successfully uninstalled setuptools-80.9.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [setuptools]2 [setuptools]
  Using cached spacy-3.8.11-cp311-cp311-macosx_11_0_arm64.whl.metadata (27 kB

## Wikidata and its SPARQL endpoint

Wikidata is an RDF dataset that contains knowledge about the world. Wikidata is a **knowledge graph**,
similar to the Google Knowledge Graph we discussed during the lectures (used by Google to show the
information on the right displayed when querying certain information, such as people). It works similarly to
Wikipedia, that is volunteers are inserting high-quality information, in addition to scripts that might add
some simpler data.
Check the following resources and properties present on Wikidata:
- https://www.wikidata.org/wiki/Q1
- https://www.wikidata.org/wiki/Q2
- https://www.wikidata.org/wiki/Q3
- https://www.wikidata.org/wiki/Q13 (the indexes Q1, Q2, . . . organize resources according to their
importance, but not only)
- https://www.wikidata.org/wiki/Q666
- https://www.wikidata.org/wiki/Wikidata:Database_reports/List_of_properties/all


In this lab we will follow a tutorial, which in addition to introducing Wikidata, it contains also a very
good presentation of SPARQL, the query language used for extracting data from an RDF graph:

https://www.wikidata.org/wiki/Wikidata:SPARQL_tutorial

After you completed the tutorial, check the list of query examples present here: 
https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples


**Exercise 1** Write queries to answer the following questions using Wikidata:

a) Write a query to find all humans who have the occupation of "mathematician" and whose employer was "École Polytechnique". Retrieve their names and their birth dates.

b) Find French citizens who are writers. List their notable works. Also, retrieve the geographic coordinates of their birthplace, but do not exclude writers who are missing this location data.

c) Group all French mathematicians by their city of birth. Count how many mathematicians were born in each city, and return the top 10 cities with the highest output of mathematicians.

Write the queries below using WDQS SPARQL endpoint in Python. We offer a simple example to start with. The advantage of querying Wikidata from Python is that you can more easily work with the results of a query. 

In [2]:
# Exercice 1 : save your queries here
# a)
from SPARQLWrapper import SPARQLWrapper, JSON
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery("""
SELECT ?nameLabel ?birthdate
WHERE {
  ?name wdt:P31 wd:Q5.
  ?name wdt:P569 ?birthdate.
  ?name wdt:P106 wd:Q170790.
  ?name wdt:P108 wd:Q273626.
  
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
}
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
print(results)

# b)


from SPARQLWrapper import SPARQLWrapper, JSON
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery("""
SELECT ?nameLabel ?notable_workLabel ?birthplaceLabel ?birthplaceCoordLabel
WHERE {
  ?name wdt:P31 wd:Q5.
  ?name wdt:P27 wd:Q142.
  ?name wdt:P106 wd:Q36180.
  ?name wdt:P800 ?notable_work.
  OPTIONAL {?name wdt:P19 ?birthplace.
           ?birthplace wdt:P625 ?birthplaceCoord}
  
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
}
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
print(results)

# c)

from SPARQLWrapper import SPARQLWrapper, JSON
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery("""
SELECT ?birthplaceLabel (COUNT(?human) AS ?count)
WHERE {
  ?human wdt:P31 wd:Q5.
  ?human wdt:P27 wd:Q142.
  ?human wdt:P106 wd:Q170790.
  ?human wdt:P19 ?birthplace.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en". }
}
GROUP BY ?birthplaceLabel
ORDER BY DESC(?count)
LIMIT 10
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
print(results)


{'head': {'vars': ['nameLabel', 'birthdate']}, 'results': {'bindings': [{'birthdate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1775-01-20T00:00:00Z'}, 'nameLabel': {'xml:lang': 'mul', 'type': 'literal', 'value': 'André-Marie Ampère'}}, {'birthdate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1768-03-21T00:00:00Z'}, 'nameLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Joseph Fourier'}}, {'birthdate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1789-08-21T00:00:00Z'}, 'nameLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Augustin-Louis Cauchy'}}, {'birthdate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1814-05-30T00:00:00Z'}, 'nameLabel': {'xml:lang': 'mul', 'type': 'literal', 'value': 'Eugene Catalan'}}, {'birthdate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'val

**Exercise 2** In this exercise we will looK at the paper "A network framework of cultural history" which you can find together with the files from this lab. Please read it and then come back to the exercises.

The paper maps the cultural history of humanity by tracking the migration of notable individuals over 2,000 years. By looking at where a notable person was born versus where they died, the authors calculated the "cultural center of gravity" for different eras (e.g., watching the intellectual capital of the world shift from Rome, to Paris, to New York).

We will try to reproduce the results from the paper using Wikidata. We will look at French artists and scientists from 1500 to 1900.

a) Write a query using the SPARQLWrapper Python library to fetch the raw data. Extract the birth year, birth city, birth city coordinates, death city, and death city coordinates of notable French individuals. Run the query for 100 years at a time to avoid a timeout.

b) Wikidata returns coordinates as a string: Point(lon lat).
Parse the JSON response into a Pandas DataFrame.
Use string splitting to separate the Point(lon lat) into two float columns (birth_lat, birth_lon, death_lat, death_lon).

c) To understand the trend of migration, you need to calculate how far these intellectuals traveled to die (often migrating from rural areas to cultural hubs). Implement the Haversine formula (https://en.wikipedia.org/wiki/Haversine_formula) in Python to calculate the great-circle distance between the birth and death coordinates on a sphere.

d) Group the data by Century (e.g., 1500s, 1600s, 1700s) and calculate:

 - the average migration distance: did 19th-century artists travel further from their birthplaces than 16th-century artists? 
 - do the place of birth differ significantly from the places of death? compute how many notable artists died there minus how many were actually born in each city. What does this number tell you about that city's cultural ecosystem?

In [ ]:
# Exercice 2 : Save your queries here

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import re
import math

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# -----------------------------
# (a) Raw data from Wikidata
# -----------------------------

def fetch_interval(start_year: int, end_year: int) -> pd.DataFrame:

    sparql.setQuery(f"""
    SELECT DISTINCT
      ?human
      (YEAR(?birthdate) AS ?birthYear)
      ?birthplace
      ?birthplaceLabel
      (STR(?birthplaceCoord) AS ?birthplaceCoordStr)
      ?deathplace
      ?deathplaceLabel
      (STR(?deathplaceCoord) AS ?deathplaceCoordStr)
    WHERE {{
      ?human wdt:P31 wd:Q5.
      ?human wdt:P27 wd:Q142.

      ?human wdt:P106 ?occupation.
      VALUES ?occupation {{ wd:Q170790 wd:Q901 }}.

      ?human wdt:P569 ?birthdate.
      FILTER(YEAR(?birthdate) >= {start_year} && YEAR(?birthdate) < {end_year}).

      ?human wdt:P19 ?birthplace.
      ?birthplace wdt:P625 ?birthplaceCoord.

      ?human wdt:P20 ?deathplace.
      ?deathplace wdt:P625 ?deathplaceCoord.

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en".
      }}
    }}
    """)
    results = sparql.query().convert()

    rows = []
    for b in results["results"]["bindings"]:
        rows.append(
            {
                "human": b["human"]["value"],
                "birthYear": int(b["birthYear"]["value"]),
                "birthplace": b["birthplace"]["value"],
                "birthplaceLabel": b.get("birthplaceLabel", {}).get("value"),
                "birthplaceCoordStr": b["birthplaceCoordStr"]["value"],
                "deathplace": b["deathplace"]["value"],
                "deathplaceLabel": b.get("deathplaceLabel", {}).get("value"),
                "deathplaceCoordStr": b["deathplaceCoordStr"]["value"],
            }
        )

    return pd.DataFrame(rows)

# Run for 100 years at a time (1500-1600, ..., 1800-1900)
frames = []
for start in range(1500, 1901, 100):
    end = start + 100
    frames.append(fetch_interval(start, end))

df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["human"]).reset_index(drop=True)

# -----------------------------
# (b) Parse coordinates
# -----------------------------

def point_str_to_lat_lon(point_str: str):
    m = re.match(r"Point\(([-0-9.]+)\s+([-0-9.]+)\)", point_str)
    if not m:
        return (math.nan, math.nan)
    lon = float(m.group(1))
    lat = float(m.group(2))
    return lat, lon

birth_lat_lon = df["birthplaceCoordStr"].apply(point_str_to_lat_lon)
df[["birth_lat", "birth_lon"]] = pd.DataFrame(birth_lat_lon.tolist(), index=df.index)

dth_lat_lon = df["deathplaceCoordStr"].apply(point_str_to_lat_lon)
df[["death_lat", "death_lon"]] = pd.DataFrame(dth_lat_lon.tolist(), index=df.index)

# -----------------------------
# (c) Haversine distance
# -----------------------------

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0  # km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Compute migration distance (birth -> death)
df["migration_km"] = df.apply(
    lambda r: haversine_km(r["birth_lat"], r["birth_lon"], r["death_lat"], r["death_lon"]),
    axis=1,
)

# -----------------------------
# (d) Group by century + city ecosystem
# -----------------------------

# Century bins like 1500s, 1600s, ...
df["century_start"] = (df["birthYear"] // 100) * 100
# Example: 1500 -> "1500s"
df["century"] = df["century_start"].astype(int).astype(str) + "s"

avg_by_century = df.groupby("century")["migration_km"].mean().sort_index()
print("Average migration distance by century (km):")
print(avg_by_century)

avg_19th = avg_by_century.get("1800s")
avg_16th = avg_by_century.get("1500s")
print(f"\n19th-century (1800s) avg: {avg_19th}")
print(f"16th-century (1500s) avg: {avg_16th}")

# Birth city vs death city counts (death - birth)
birth_counts = df.groupby("birthplaceLabel")["human"].nunique()
death_counts = df.groupby("deathplaceLabel")["human"].nunique()

all_cities = birth_counts.index.union(death_counts.index)
city_df = pd.DataFrame(index=all_cities)
city_df["births"] = birth_counts.reindex(all_cities).fillna(0).astype(int)
city_df["deaths"] = death_counts.reindex(all_cities).fillna(0).astype(int)
city_df["deaths_minus_births"] = city_df["deaths"] - city_df["births"]

print("\nCities where more people died than were born (top 10, positive diff):")
print(city_df.sort_values("deaths_minus_births", ascending=False).head(10))

print("\nCities where more people were born than died (top 10, negative diff):")
print(city_df.sort_values("deaths_minus_births", ascending=True).head(10))


Fetching [1500, 1600)...
Fetching [1600, 1700)...
Fetching [1700, 1800)...
Fetching [1800, 1900)...
Fetching [1900, 2000)...
Average migration distance by century (km):
century
1500s     288.063797
1600s     695.956424
1700s     567.528983
1800s     451.670420
1900s    1174.977514
Name: migration_km, dtype: float64

19th-century (1800s) avg: 451.67041992139224
16th-century (1500s) avg: 288.0637967299043

Cities where more people died than were born (top 10, positive diff):
                              births  deaths  deaths_minus_births
Paris                            117     280                  163
14th arrondissement of Paris      10      25                   15
13th arrondissement of Paris       2      15                   13
15th arrondissement of Paris       7      20                   13
5th arrondissement of Paris        1       7                    6
Villejuif                          0       6                    6
Nice                               2       8                

## Similarity functions for strings in Python

**Exercise 3**
Compare "Lillois" against the following variations:

    "lillois" (case variation)

    "Lilloise" (gender variation)

    "Lillios" (transposition typo)

    "LilIois" (OCR Error: capital 'I' instead of lowercase 'l')

    "Lillwa" (phonetic misspelling)


For each pair, calculate the following metrics using jellyfish https://www.jpt.sh/projects/jellyfish/:

- Levenshtein Distance

- Damerau-Levenshtein Distance 

- Jaro-Winkler Similarity 

- Soundex (A phonetic algorithm) 

Compare the results you obtain and note which algorithm performs best in which setting.


In [4]:
# Solution exercise 3
import sys
import jellyfish
import pandas as pd

base = "Lillois"
variations = [
    ("lillois", "case variation"),
    ("Lilloise", "gender variation"),
    ("Lillios", "transposition typo"),
    ("LilIois", "OCR Error"),
    ("Lillwa", "phonetic misspelling"),
]

soundex_base = jellyfish.soundex(base)

rows = []
for var, setting in variations:
    lev = jellyfish.levenshtein_distance(base, var)
    damerau_lev = jellyfish.damerau_levenshtein_distance(base, var)
    jaro_winkler = jellyfish.jaro_winkler_similarity(base, var)
    soundex_var = jellyfish.soundex(var)

    rows.append(
        {
            "setting": setting,
            "variation": var,
            "levenshtein_distance": lev,
            "damerau_levenshtein_distance": damerau_lev,
            "jaro_winkler_similarity": jaro_winkler,
            "soundex_base": soundex_base,
            "soundex_variation": soundex_var,
            "soundex_match": soundex_var == soundex_base,
        }
    )

df = pd.DataFrame(rows)

# Show results
print(df)

                setting variation  levenshtein_distance  \
0        case variation   lillois                     1   
1      gender variation  Lilloise                     1   
2    transposition typo   Lillios                     2   
3             OCR Error   LilIois                     1   
4  phonetic misspelling    Lillwa                     3   

   damerau_levenshtein_distance  jaro_winkler_similarity soundex_base  \
0                             1                 0.849206         L420   
1                             1                 0.975000         L420   
2                             1                 0.971429         L420   
3                             1                 0.933333         L420   
4                             3                 0.847619         L420   

  soundex_variation  soundex_match  
0              L420           True  
1              L420           True  
2              L420           True  
3              L420           True  
4              L400  

**Exercise 4**
We have talked today about TF-IDF document representation. Recall that TF-IDF represents a document as a sparse vector in a high-dimensional space where each dimension corresponds to a unique word in the entire vocabulary. It calculates the relevance of a term t to a document d within a corpus D by multiplying how often the word appears in the document (TF) by the logarithmically scaled inverse fraction of the documents that contain the word (IDF). 

However, word embeddings are more frequently used today. 
Word embeddings map words from a discrete vocabulary to dense, low-dimensional, continuous real-valued vectors (e.g., placing them in a continuous vector space). They work because of the distributional hypothesis—the linguistic theory that words occurring in similar contexts share similar meanings. We compute them using neural networks. A document embedding is the average (weighted) of the words contained in it.

Properties: Word embeddings capture latent semantic and syntactic relationships. Because vectors are positioned in a metric space based on context, synonyms (like "cloud" and "sky") will have a high cosine similarity even if they share zero characters.

Please check the spacy library and in particular the part on word embeddings https://spacy.io/usage/linguistic-features#vectors-similarity . Note that vector embeddings are language dependent, so if you are working with English, you need to download and use the English package (*python -m spacy download en_core_web_lg*), while for French you need to use *python -m spacy download fr_core_news_lg*. 

Write a Python script to compute the spaCy vector similarity (which calculates the cosine similarity of the underlying word embeddings) for the following three pairs of strings.

a) Synonyms: "Tech Corporation" and "Technology Company"

b) String with high token overlap: "Goldman Sachs Group International, London", "Morgan Stanley Group International, 
London","Goldman Sachs"

c) Antonyms:"I love this product", "I hate this product"

What do you observe?

In [12]:
"""Solution exercise 4

On calcule la similarité de spaCy (cosinus entre les vecteurs) entre des paires de chaînes.
"""

import spacy
pairs = {
    "a_synonyms": ("Tech Corporation", "Technology Company"),
    # Le texte de l'exercice dans le notebook a une petite coupure/quote dans la ligne b).
    # L'intention la plus cohérente est la comparaison de deux acteurs financiers à Londres.
    "b_token_overlap": (
        "Goldman Sachs Group International, London",
        "Morgan Stanley Group International, London",
    ),
    "c_antonyms": ("I love this product", "I hate this product"),
}


def load_model_with_vectors():
    # On tente en priorité l'anglais (les textes de l'exercice sont en anglais).
    candidates = ["en_core_web_lg"]
    last_err = None
    for name in candidates:
        nlp = spacy.load(name)
        # Indice simple que le modèle contient bien des vecteurs.
        if getattr(nlp.vocab, "vectors_length", 0) and nlp.vocab.vectors_length > 0:
            return nlp

    raise OSError(
        "Aucun modèle spaCy avec vecteurs n'a pu être chargé. "
        "Installe par exemple : `python -m spacy download en_core_web_lg`"
        + (f"\nDétail: {last_err}" if last_err else "")
    )


nlp = load_model_with_vectors()

print("spaCy vector similarities (cosine):")
for label, (s1, s2) in pairs.items():
    print(s1, s2)
    doc1 = nlp(s1)
    doc2 = nlp(s2)
    sim = doc1.similarity(doc2)
    print(f"- {label}: {sim:.4f}")


spaCy vector similarities (cosine):
Tech Corporation Technology Company
- a_synonyms: 0.7169
Goldman Sachs Group International, London Morgan Stanley Group International, London
- b_token_overlap: 0.8746
I love this product I hate this product
- c_antonyms: 0.9552


## Similarity between Entities

Let's look at how to match entities from different files.
The goal is not to implement a particular algorithm, but that you propose your method in the context of our datasets.

We propose to link the entities of two citation databases from two well-known websites  DBLP http://dblp.uni-trier.de and ACM https://dl.acm.org/ .
You can find the files on the website https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution at the following address http://dbs.uni-leipzig.de/file/DBLP-ACM.zip .
In this zip file, you will also find correct matches to compare your approach to the truth.

To read these files, you can use the csv library of Python or pandas as you have seen in previous labs. Please note that you might need to use the encoding *latin-1*.

**Exercise 5.**
Propose an algorithm to compute similar entities in the two files. Try also blocking techniques to speed up the algorithm. Compare your results with the correct matches file and compute the different measures (precision, recall, F1) to evaluate your algorithm.

In [28]:
import os
import re
import math
import json
import zipfile
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# ----------------------------
# 1) Load dataset
# ----------------------------
DATA_URL = "http://dbs.uni-leipzig.de/file/DBLP-ACM.zip"
DATA_DIR = "data_dblp_acm"
ZIP_PATH = os.path.join(DATA_DIR, "DBLP-ACM.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "extracted")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(EXTRACT_DIR) or not any(f.endswith(".csv") for f in os.listdir(EXTRACT_DIR)):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    if not os.path.exists(ZIP_PATH):
        try:
            import requests
        except Exception:
            raise RuntimeError("Missing 'requests'. Please install it (pip install requests) and rerun.")

        print(f"Downloading dataset from {DATA_URL}...")
        r = requests.get(DATA_URL, timeout=60)
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            f.write(r.content)

    print("Extracting dataset zip...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

# Find CSV paths
csv_paths = []
for root, _, files in os.walk(EXTRACT_DIR):
    for fn in files:
        if fn.lower().endswith(".csv"):
            csv_paths.append(os.path.join(root, fn))

if not csv_paths:
    raise FileNotFoundError("No CSV files found in the extracted dataset directory.")

# Heuristics: DBLP/ACM + perfect mapping

def pick_csv(needles, exclude=None):
    exclude = exclude or []
    candidates = []
    for p in csv_paths:
        name = os.path.basename(p).lower()
        if all(n.lower() in name for n in needles) and not any(e.lower() in name for e in exclude):
            candidates.append(p)
    return sorted(candidates, key=len)[0] if candidates else None

perfect_path = pick_csv(["perfect"], exclude=["amazon"])
if perfect_path is None:
    perfect_path = pick_csv(["mapping"], exclude=["amazon"])
if perfect_path is None:
    perfect_path = pick_csv(["perfectmapping"] , exclude=["amazon"])
if perfect_path is None:
    # Fallback: any csv containing 'ACM' and 'DBLP' and 'perfect' like substring
    perfect_candidates = [p for p in csv_paths if ("perfect" in os.path.basename(p).lower())]
    perfect_path = sorted(perfect_candidates)[0] if perfect_candidates else None

dblp_path = pick_csv(["dblp"])
acm_path = pick_csv(["acm" ])

# Some archives use DBLP.csv and ACM.csv without obvious 'dblp' substring in mapping.
# Ensure we didn't pick the mapping file as dblp/acm.
if dblp_path and perfect_path and os.path.abspath(dblp_path) == os.path.abspath(perfect_path):
    dblp_path = None
if acm_path and perfect_path and os.path.abspath(acm_path) == os.path.abspath(perfect_path):
    acm_path = None

# Final fallbacks
if dblp_path is None:
    dblp_candidates = [p for p in csv_paths if "dblp" in os.path.basename(p).lower() and "perfect" not in os.path.basename(p).lower() and "mapping" not in os.path.basename(p).lower()]
    dblp_path = sorted(dblp_candidates)[0] if dblp_candidates else None

if acm_path is None:
    acm_candidates = [p for p in csv_paths if "acm" in os.path.basename(p).lower() and "perfect" not in os.path.basename(p).lower() and "mapping" not in os.path.basename(p).lower()]
    acm_path = sorted(acm_candidates)[0] if acm_candidates else None

if perfect_path is None or dblp_path is None or acm_path is None:
    print("Found CSVs:")
    for p in sorted(csv_paths):
        print(" -", p)
    raise RuntimeError("Could not identify DBLP/ACM/perfect mapping CSV files automatically. Please check file names in the extracted dataset.")

print("Using files:")
print(" - DBLP:", dblp_path)
print(" - ACM:", acm_path)
print(" - Perfect mapping:", perfect_path)

# Read data
# The lab prompt suggests 'latin-1'.

def read_csv(path):
    return pd.read_csv(path, encoding="latin-1")

dblp_df = read_csv(dblp_path)
acm_df = read_csv(acm_path)
truth_df = read_csv(perfect_path)

print("Loaded sizes:", len(dblp_df), len(acm_df), len(truth_df))

# ----------------------------
# 2) Column detection
# ----------------------------

def find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    # substring fallback
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

# IDs
id_dblp = find_col(dblp_df, ["id", "dblp_id", "dblpId", "dbid", "DBLP_id"])
if id_dblp is None:
    id_dblp = find_col(dblp_df, ["ID"])

id_acm = find_col(acm_df, ["id", "acm_id", "acmId", "acmid", "ACM_id"])
if id_acm is None:
    id_acm = find_col(acm_df, ["ID"])

# Textual attributes
title_dblp = find_col(dblp_df, ["title", "name", "paper_title", "booktitle", "venue_title"])
authors_dblp = find_col(dblp_df, ["authors", "author", "full_authors", "authorString"])
venue_dblp = find_col(dblp_df, ["venue", "journal", "conference", "booktitle"])
year_dblp = find_col(dblp_df, ["year", "publication_year"])

title_acm = find_col(acm_df, ["title", "name", "paper_title", "booktitle", "venue_title"])
authors_acm = find_col(acm_df, ["authors", "author", "full_authors", "authorString"])
venue_acm = find_col(acm_df, ["venue", "journal", "conference", "booktitle"])
year_acm = find_col(acm_df, ["year", "publication_year"])

missing = [
    ("dblp_id", id_dblp), ("dblp_title", title_dblp), ("dblp_authors", authors_dblp),
    ("acm_id", id_acm), ("acm_title", title_acm), ("acm_authors", authors_acm),
]
missing = [name for name, col in missing if col is None]
if missing:
    raise RuntimeError("Missing expected columns in DBLP/ACM: " + ", ".join(missing) + ".\n" +
                       f"dblp columns: {list(dblp_df.columns)}\nacm columns: {list(acm_df.columns)}")

# Truth pairs detection
# We want pairs (acm_id, dblp_id) or vice versa.

def detect_truth_pair_cols(df):
    cols_lower = {c.lower(): c for c in df.columns}
    dblp_col = None
    acm_col = None
    for c in df.columns:
        cl = c.lower()
        if "dblp" in cl or "dbid" in cl:
            dblp_col = c
        if "acm" in cl or "acmid" in cl:
            acm_col = c

    if dblp_col and acm_col:
        return acm_col, dblp_col

    # substring fallback
    for c in df.columns:
        cl = c.lower()
        if dblp_col is None and ("dblp" in cl or "dbid" in cl):
            dblp_col = c
        if acm_col is None and ("acm" in cl or "acmid" in cl):
            acm_col = c

    if dblp_col and acm_col:
        return acm_col, dblp_col

    # Fallback: take first 2 columns
    if len(df.columns) < 2:
        raise RuntimeError("Perfect mapping file has fewer than 2 columns.")
    return df.columns[1], df.columns[0]

truth_acm_col, truth_dblp_col = detect_truth_pair_cols(truth_df)

# Cast IDs to strings for robust set comparisons

def to_str(x):
    if pd.isna(x):
        return ""
    # Avoid scientific notation for large numeric IDs
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    return str(x).strip()

truth_pairs = set((to_str(a), to_str(d)) for a, d in zip(truth_df[truth_acm_col], truth_df[truth_dblp_col]))
print("Ground truth pairs:", len(truth_pairs))

# ----------------------------
# 3) Preprocessing helpers
# ----------------------------

STOPWORDS = {
    "the", "a", "an", "of", "and", "or", "to", "for", "in", "on", "at", "by", "from",
    "with", "without", "using", "and", "as", "is", "are", "was", "were", "be", "been",
    "it", "this", "that"
}


def strip_accents(s):
    # NFKD decomposition + drop combining chars to get ASCII.
    s = unicodedata.normalize('NFKD', str(s))
    return ''.join(ch for ch in s if not unicodedata.combining(ch))


def normalize_str(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = strip_accents(s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def tokenize(s):
    s = normalize_str(s)
    if not s:
        return []
    toks = [t for t in s.split(" ") if t and t not in STOPWORDS]
    return toks


def make_trigrams(s):
    s = normalize_str(s)
    s = re.sub(r"\s+", "", s)  # remove spaces for char ngrams
    if len(s) < 3:
        return set([s]) if s else set()
    return {s[i:i+3] for i in range(len(s) - 2)}


def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0


def parse_last_names(authors_value):
    if authors_value is None or (isinstance(authors_value, float) and math.isnan(authors_value)):
        return []
    s = strip_accents(str(authors_value))
    # Common separators: ';', ',', ' and ', '&'
    parts = re.split(r"\s*;\s*|\s*&\s*|\s+and\s+|\s*,\s*(?=[A-Z])", s)
    # Fallback: if no split happened, still try to split by ';'
    if len(parts) == 1:
        parts = re.split(r"\s*;\s*|\s*&\s*|\s+and\s+", s)

    last_names = []
    for p in parts:
        p = normalize_str(p)
        if not p:
            continue
        toks = p.split(" ")
        if not toks:
            continue
        # Heuristic: take last token; if format is 'last, first', take token before comma already removed.
        last_names.append(toks[-1])

    # Deduplicate but keep deterministic ordering
    uniq = []
    seen = set()
    for ln in last_names:
        if ln not in seen:
            seen.add(ln)
            uniq.append(ln)
    return uniq


def venue_tokens(venue_value):
    toks = tokenize(venue_value)
    if not toks:
        return []
    # Keep only first few tokens to reduce noise.
    return toks[:6]


def year_bucket(year_value):
    if year_value is None or (isinstance(year_value, float) and math.isnan(year_value)):
        return "NA"
    try:
        y = int(float(year_value))
        return str(y)
    except Exception:
        return "NA"

# ----------------------------
# 4) Build blocking index on DBLP
# ----------------------------

# Precompute features for speed

def precompute(df, id_col, title_col, authors_col, venue_col, year_col):
    feats = {}
    for _, row in df.iterrows():
        rid = to_str(row[id_col])
        title_val = row[title_col] if title_col else ""
        authors_val = row[authors_col] if authors_col else ""
        venue_val = row[venue_col] if venue_col else ""
        year_val = row[year_col] if year_col else np.nan

        toks = tokenize(title_val)
        trig = make_trigrams(title_val)
        last_names = parse_last_names(authors_val)
        v_toks = venue_tokens(venue_val)
        yb = year_bucket(year_val)

        first_tok = toks[0] if toks else ""
        ln0 = last_names[0] if last_names else ""

        # Blocking keys (unioned): title first token + year, author first last name + year
        keys = set()
        if first_tok:
            keys.add(f"T|{first_tok}|Y|{yb}")
        if ln0:
            keys.add(f"A|{ln0}|Y|{yb}")

        feats[rid] = {
            "title_toks": set(toks),
            "title_trigrams": trig,
            "authors_last": set(last_names),
            "venue_toks": set(v_toks),
            "year": yb,
            "keys": keys,
            "title_tokens_list": toks,
            "year_raw": yb,
        }
    return feats

print("Precomputing features...")

dblp_feats = precompute(dblp_df, id_dblp, title_dblp, authors_dblp, venue_dblp, year_dblp)
acm_feats = precompute(acm_df, id_acm, title_acm, authors_acm, venue_acm, year_acm)

# Blocking index
blocking = defaultdict(list)  # key -> list of dblp ids
for did, f in dblp_feats.items():
    for k in f["keys"]:
        blocking[k].append(did)

print("Blocking keys:", len(blocking))

# ----------------------------
# 5) Similarity + prediction
# ----------------------------

def title_similarity(fa, fb):
    # Mix trigram Jaccard + token-set Jaccard.
    t1 = jaccard(fa["title_trigrams"], fb["title_trigrams"])
    t2 = jaccard(fa["title_toks"], fb["title_toks"])
    return 0.6 * t1 + 0.4 * t2


def authors_similarity(fa, fb):
    return jaccard(fa["authors_last"], fb["authors_last"])


def venue_similarity(fa, fb):
    return jaccard(fa["venue_toks"], fb["venue_toks"])


def year_similarity(fa, fb):
    return 1.0 if fa["year"] != "NA" and fa["year"] == fb["year"] else 0.0


def combined_score(fa, fb, w=(0.6, 0.2, 0.1, 0.1)):
    w_title, w_auth, w_year, w_venue = w
    return (
        w_title * title_similarity(fa, fb)
        + w_auth * authors_similarity(fa, fb)
        + w_year * year_similarity(fa, fb)
        + w_venue * venue_similarity(fa, fb)
    )

# Evaluate with top-1 per ACM + optional one-to-one greedy resolution

THRESHOLD = 0.55
TOP_CANDIDATES_PER_BLOCK = 80  # limit to avoid too many comparisons

pred_list = []  # (score, acm_id, dblp_id)

for acm_id, fa in acm_feats.items():
    # gather candidates from blocks
    cand_ids = set()
    for k in fa["keys"]:
        for did in blocking.get(k, []):
            cand_ids.add(did)

    if not cand_ids:
        continue

    # Quick pruning: keep top by title token Jaccard (cheap)
    # (If candidate set is already small, this is harmless.)
    scored_quick = []
    ta = fa["title_toks"]
    for did in cand_ids:
        fb = dblp_feats[did]
        quick = jaccard(ta, fb["title_toks"])
        scored_quick.append((quick, did))
    scored_quick.sort(reverse=True, key=lambda x: x[0])

    top_dids = [did for _, did in scored_quick[:TOP_CANDIDATES_PER_BLOCK]]

    best = (0.0, None)
    for did in top_dids:
        fb = dblp_feats[did]
        s = combined_score(fa, fb)
        if s > best[0]:
            best = (s, did)

    best_score, best_did = best
    if best_did is not None and best_score >= THRESHOLD:
        pred_list.append((best_score, acm_id, best_did))

print("Raw predictions (after threshold):", len(pred_list))

# One-to-one greedy resolution: prefer highest score pairs
pred_list.sort(reverse=True, key=lambda x: x[0])
used_acm = set()
used_dblp = set()
pred_pairs = set()

for score, a_id, d_id in pred_list:
    if a_id in used_acm or d_id in used_dblp:
        continue
    used_acm.add(a_id)
    used_dblp.add(d_id)
    pred_pairs.add((a_id, d_id))

print("Predictions after one-to-one resolution:", len(pred_pairs))

# ----------------------------
# 6) Metrics against ground truth
# ----------------------------

TP = len(pred_pairs & truth_pairs)
FP = len(pred_pairs - truth_pairs)
FN = len(truth_pairs - pred_pairs)

precision = TP / (TP + FP) if (TP + FP) else 0.0
recall = TP / (TP + FN) if (TP + FN) else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print("--- Evaluation ---")
print(f"TP={TP} FP={FP} FN={FN}")
print(f"Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}")

# Optional: show a few correct and incorrect examples

def find_rows(df, id_col, id_value):
    # small helper for display
    mask = df[id_col].astype(str) == str(id_value)
    if mask.any():
        return df[mask].head(1)
    return None

correct = list(pred_pairs & truth_pairs)[:3]
wrong = list(pred_pairs - truth_pairs)[:3]

print("\nSample correct pairs (acm_id, dblp_id):", correct)
print("Sample wrong predicted pairs (acm_id, dblp_id):", wrong)

# If available, print titles for samples (helps for debugging)
for a_id, d_id in correct:
    a_row = find_rows(acm_df, id_acm, a_id)
    d_row = find_rows(dblp_df, id_dblp, d_id)
    if a_row is not None and d_row is not None:
        print("\nCorrect:")
        print(" ACM title:", str(a_row.iloc[0][title_acm])[:120])
        print(" DBLP title:", str(d_row.iloc[0][title_dblp])[:120])


Extracting dataset zip...
Using files:
 - DBLP: data_dblp_acm/extracted/DBLP2.csv
 - ACM: data_dblp_acm/extracted/ACM.csv
 - Perfect mapping: data_dblp_acm/extracted/DBLP-ACM_perfectMapping.csv
Loaded sizes: 2616 2294 2224
Ground truth pairs: 2224
Precomputing features...
Blocking keys: 4166
Raw predictions (after threshold): 2204
Predictions after one-to-one resolution: 2184
--- Evaluation ---
TP=2174 FP=10 FN=50
Precision=0.9954  Recall=0.9775  F1=0.9864

Sample correct pairs (acm_id, dblp_id): [('335414', 'conf/sigmod/CorralMTV00'), ('672211', 'conf/vldb/DayalHL01'), ('375751', 'conf/sigmod/LahiriG01')]
Sample wrong predicted pairs (acm_id, dblp_id): [('601865', 'journals/sigmod/Aberer02a'), ('641000', 'journals/sigmod/RossNO03'), ('505049', 'journals/tods/Snodgrass01a')]

Correct:
 ACM title: Closest pair queries in spatial databases
 DBLP title: Closest Pair Queries in Spatial Databases

Correct:
 ACM title: Business Process Coordination: State of the Art, Trends, and Open Issues


Advanced Entity Resolution on E-commerce Data

In the previous exercises, you matched bibliographic records using mostly short string comparisons. In real-world scenarios, datasets are highly heterogeneous. You will now match product listings from Amazon and Google.

You can download the dataset here: https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip
Inside, you will find Amazon.csv, GoogleProducts.csv, and the perfect matches in Amzon_GoogleProducts_perfectMapping.csv.


**Exercise 6** Write a function compute_similarity(row_amazon, row_google) that computes a combined score based on multiple rules:

- name similarity

- description similarity

-  price similarity

Try different weights for each component in the score and then compute the F1 score of the match using the Amzon_GoogleProducts_perfectMapping.csv

In [29]:
from tp2_ex6 import compute_similarity, run_exercise6

# Lance l'exercice 6 (téléchargement dataset si besoin + recherche de pondérations + F1)
run_exercise6()

Extracting dataset...
weights (name, desc, price)=(0.60, 0.30, 0.10) -> precision=0.670 recall=0.574 F1=0.618
weights (name, desc, price)=(0.50, 0.40, 0.10) -> precision=0.666 recall=0.570 F1=0.614
weights (name, desc, price)=(0.50, 0.30, 0.20) -> precision=0.670 recall=0.574 F1=0.618
weights (name, desc, price)=(0.40, 0.50, 0.10) -> precision=0.660 recall=0.565 F1=0.609
weights (name, desc, price)=(0.40, 0.40, 0.20) -> precision=0.665 recall=0.569 F1=0.613
weights (name, desc, price)=(0.30, 0.60, 0.10) -> precision=0.649 recall=0.555 F1=0.598

Best result:
weights=(0.6000000000000001, 0.30000000000000004, 0.10000000000000002) precision=0.670 recall=0.574 F1=0.618


{'weights': (0.6000000000000001, 0.30000000000000004, 0.10000000000000002),
 'precision': 0.6702605570530099,
 'recall': 0.5738461538461539,
 'f1': 0.6183174471612102}